# Explore GHTorrent Tables for Sentiment Mapping (Notebook 2)

This notebook explores the GHTorrent database to understand where the 7,122 Gold Standard sentiment comments are stored and how they connect to real projects.

The goal is to answer which projects have sentiment-labeled comments and whether those comments are reachable from a canonical (non-fork) repository.

### Planned Output
By the end of this notebook, you should have:
1. A breakdown of how sentiment comments are split between commit comment and PR comment tables
2. A ranked list of projects with the most sentiment-labeled commit comments
3. A ranked list of projects with the most sentiment-labeled PR comments
4. A global summary of how many sentiment comments are reachable via canonical repos vs. forks only

### Step 1: Import dependencies and connect to MySQL

We reuse the same connection pattern as Notebook 1. Update the credentials below to match your local MySQL setup.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
MYSQL_HOST     = "localhost"
MYSQL_PORT     = 3306
MYSQL_USER     = "root"
MYSQL_PASSWORD = "password"
MYSQL_DB       = "github"

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)
print("Connected to MySQL.")

Connected to MySQL.


### Check 1: How sentiment comments are distributed across tables

There are two kinds of Github comments in GHTorrent: commit comments (discussions posted about commits) and PR inline comments (left on a specific line of code in a pull request).

The Gold Standard dataset includes both types. The same `ID` value maps to `comment_id` in both `commit_comments` and `pull_request_comments`.

This check tells us how many sentiment IDs join to each table. Some IDs appear in both tables (overlap = 85), meaning a small number of comments were captured under both endpoints in GHTorrent. The total unique IDs should sum to 7,122.

Expected values:
- Commit comment matches: ~4,317
- PR comment matches: ~2,890
- Overlap (both): ~85
- Commit-only: 4,232 | PR-only: 2,805 | Total unique: 7,122

In [ ]:
with engine.connect() as con:
    commit_count = pd.read_sql(text("""
        SELECT COUNT(*) AS commit_comment_matches
        FROM comment_sentiment s
        INNER JOIN commit_comments cc ON s.ID = cc.comment_id;
    """), con).iloc[0, 0]

    pr_count = pd.read_sql(text("""
        SELECT COUNT(*) AS pr_comment_matches
        FROM comment_sentiment s
        INNER JOIN pull_request_comments prc ON s.ID = prc.comment_id;
    """), con).iloc[0, 0]

    overlap = pd.read_sql(text("""
        SELECT COUNT(*) AS overlap
        FROM comment_sentiment s
        INNER JOIN commit_comments cc ON s.ID = cc.comment_id
        INNER JOIN pull_request_comments prc ON s.ID = prc.comment_id;
    """), con).iloc[0, 0]

commit_only  = commit_count - overlap
pr_only      = pr_count - overlap
total_unique = commit_only + pr_only + overlap

summary = pd.DataFrame({
    'Category': ['Commit matches', 'PR matches', 'Overlap (both)', 'Commit-only', 'PR-only', 'Total unique'],
    'Count':    [commit_count, pr_count, overlap, commit_only, pr_only, total_unique],
    'Expected': [4317, 2890, 85, 4232, 2805, 7122]
})
display(summary)

if total_unique == 7122:
    print("PASS: total unique IDs = 7122.")
else:
    print(f"WARNING: total unique IDs = {total_unique}, expected 7122.")

### Check 2: Projects with the most sentiment-labeled commit comments

This query ranks projects by how many of their commit comments are in the Gold Standard. This tells us which projects are most heavily represented in the labeled data.

In [ ]:
query_check2 = """
SELECT
    p.id                         AS project_id,
    p.name                       AS project_name,
    p.url                        AS project_url,
    COUNT(DISTINCT s.ID)         AS labeled_comment_count
FROM projects p
INNER JOIN commits           c   ON p.id = c.project_id
INNER JOIN commit_comments   cc  ON c.id = cc.commit_id
INNER JOIN comment_sentiment s   ON cc.comment_id = s.ID
GROUP BY p.id, p.name, p.url
ORDER BY labeled_comment_count DESC;
"""

with engine.connect() as con:
    check2 = pd.read_sql(text(query_check2), con)

print(f"Projects with sentiment-labeled commit comments: {len(check2)}")
display(check2)

### Check 3: Projects with the most sentiment-labeled PR comments

Same ranking, but for pull request inline comments. PR inline comments are joined through `pull_requests` via `base_repo_id`. The base repo is the canoonical project the PR was opened in.

In [ ]:
query_check3 = """
SELECT
    p.id                         AS project_id,
    p.name                       AS project_name,
    p.url                        AS project_url,
    COUNT(DISTINCT s.ID)         AS labeled_comment_count
FROM projects p
INNER JOIN pull_requests         pr  ON p.id = pr.base_repo_id
INNER JOIN pull_request_comments prc ON pr.id = prc.pull_request_id
INNER JOIN comment_sentiment     s   ON prc.comment_id = s.ID
GROUP BY p.id, p.name, p.url
ORDER BY labeled_comment_count DESC;
"""

with engine.connect() as con:
    check3 = pd.read_sql(text(query_check3), con)

print(f"Projects with sentiment-labeled PR comments: {len(check3)}")
display(check3)

### Check 4: Canonical repo vs fork accessibility

GitHub projects get forked frequently. A fork shares the same commit history as its upstream repo, which means the same comment IDs can appear under both the original (canonical) project and one or more forks in GHTorrent.

This matters for Notebook 3. When we generate Kaiaulu config files, we only want to target canonical repos. Downloading from a fork is redundant since the canonical repo already contains all the same commits.

Expected values:
- `canonical_only`: ~4,555
- `fork_only`: ~569 (these will be missed when targeting canonical repos only)
- `both_sides`: ~2,083
- Canonical reachable rate: ~92.1%

In [9]:
query_check4 = """
WITH RECURSIVE project_root AS (
    SELECT p.id AS project_id, p.id AS root_id
    FROM projects p
    WHERE p.forked_from IS NULL
    UNION ALL
    SELECT c.id AS project_id, pr.root_id
    FROM projects c
    JOIN project_root pr ON c.forked_from = pr.project_id
),
comment_project_rows AS (
    SELECT cs.ID AS comment_id, c.project_id, 'commit_comment' AS source_tag
    FROM comment_sentiment cs
    JOIN commit_comments cc ON cs.ID = cc.comment_id
    JOIN commits c ON cc.commit_id = c.id
    UNION ALL
    SELECT cs.ID AS comment_id, pr.base_repo_id AS project_id, 'pr_comment' AS source_tag
    FROM comment_sentiment cs
    JOIN pull_request_comments prc ON cs.ID = prc.comment_id
    JOIN pull_requests pr ON prc.pull_request_id = pr.id
    UNION ALL
    SELECT cs.ID AS comment_id, pr.head_repo_id AS project_id, 'pr_comment' AS source_tag
    FROM comment_sentiment cs
    JOIN pull_request_comments prc ON cs.ID = prc.comment_id
    JOIN pull_requests pr ON prc.pull_request_id = pr.id
),
labeled AS (
    SELECT
        cpr.comment_id,
        cpr.source_tag,
        pr.root_id,
        (cpr.project_id = pr.root_id) AS is_canonical
    FROM comment_project_rows cpr
    JOIN project_root pr ON pr.project_id = cpr.project_id
),
comment_flags AS (
    SELECT
        root_id, source_tag, comment_id,
        MAX(CASE WHEN is_canonical     THEN 1 ELSE 0 END) AS has_canonical,
        MAX(CASE WHEN NOT is_canonical THEN 1 ELSE 0 END) AS has_fork
    FROM labeled
    GROUP BY root_id, source_tag, comment_id
),
global_counts AS (
    SELECT
        COUNT(*)                                                             AS mapped_comment_ids,
        SUM(CASE WHEN has_canonical = 1 AND has_fork = 0 THEN 1 ELSE 0 END) AS canonical_only,
        SUM(CASE WHEN has_canonical = 0 AND has_fork = 1 THEN 1 ELSE 0 END) AS fork_only,
        SUM(CASE WHEN has_canonical = 1 AND has_fork = 1 THEN 1 ELSE 0 END) AS both_sides
    FROM comment_flags
)
SELECT
    canonical_only,
    fork_only,
    both_sides,
    ROUND(100 * fork_only / NULLIF(mapped_comment_ids, 0), 2)                     AS fork_only_pct,
    ROUND(100 * (canonical_only + both_sides) / NULLIF(mapped_comment_ids, 0), 2) AS canonical_reachable_pct
FROM global_counts;
"""

with engine.connect() as con:
    check4 = pd.read_sql(text(query_check4), con)

print("Canonical vs fork accessibility summary:")
print(f"  canonical_only        : {check4['canonical_only'].iloc[0]}  (expected ~4555)")
print(f"  fork_only             : {check4['fork_only'].iloc[0]}  (expected ~569)")
print(f"  both_sides            : {check4['both_sides'].iloc[0]}  (expected ~2083)")
print(f"  fork_only_pct         : {check4['fork_only_pct'].iloc[0]}%  (expected ~7.9%)")
print(f"  canonical_reachable % : {check4['canonical_reachable_pct'].iloc[0]}%  (expected ~92.1%)")
display(check4)

Canonical vs fork accessibility summary:
  canonical_only        : 4555.0  (expected ~4555)
  fork_only             : 569.0  (expected ~569)
  both_sides            : 2083.0  (expected ~2083)
  fork_only_pct         : 7.9%  (expected ~7.9%)
  canonical_reachable % : 92.1%  (expected ~92.1%)


,canonical_only,fork_only,both_sides,fork_only_pct,canonical_reachable_pct
0,4555.0,569.0,2083.0,7.9,92.1


### When to move on to Notebook 3

Move to Notebook 3 when all of the following are true:

1. Check 1: PASS printed and total unique IDs = 7,122
2. Check 2: returns a non-empty DataFrame of projects with commit comment matches
3. Check 3: returns a non-empty DataFrame of projects with PR comment matches
4. Check 4: runs without error and shows a non-zero `canonical_reachable_pct`

If any check returns zero rows, the most likely cause is that `comment_sentiment` was not loaded correctly in Notebook 1. Re-run Notebook 1 first.

The ~7.9% of comments that are `fork_only` will be missed when we target only canonical repos in Notebook 3. This is an acceptable tradeoff. We document it here so the limitation is visible.